In [ ]:
import os
print(os.path.expanduser("~"))

In [3]:
import os
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
import pickle

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/macbook_air/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/macbook_air/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/macbook_air/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [4]:
LYRICS_DIR = os.path.expanduser("~/music4all_data/lyrics/")

records = []
for filename in os.listdir(LYRICS_DIR):
    if filename.endswith(".txt"):
        song_id = filename.replace(".txt", "")
        filepath = os.path.join(LYRICS_DIR, filename)
        with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
            lyrics = f.read().strip()
        if lyrics:
            records.append({"song_id": song_id, "lyrics": lyrics})

df = pd.DataFrame(records)
print(f"Loaded {len(df)} songs with lyrics")
df.head()

Loaded 109269 songs with lyrics


,song_id,lyrics
0,QPozpoSeoWPWUp5k,"You and me together, stars forever\nYou and me..."
1,7AbZpDQ5co4eiTyc,The last train is nearly due\nThe Underground ...
2,SazEmQ5Ls4WqhvGE,All it takes is the main attraction\nScattered...
3,mcSyQWsHtskCQIBJ,Turn the lights out\nLock the door\nFan the fl...
4,LF3ygFyHjEMefIXc,INSTRUMENTAL


In [8]:
import re
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

def clean_lyrics(text):
    # Remove instrumental tags
    if text.strip().upper() == "INSTRUMENTAL":
        return None
    # Remove [Verse], [Chorus], [Bridge] etc.
    text = re.sub(r'\[.*?\]', '', text)
    # Lowercase
    text = text.lower()
    # Remove punctuation and numbers
    text = re.sub(r'[^a-z\s]', '', text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and short tokens
    tokens = [t for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['cleaned_lyrics'] = df['lyrics'].apply(clean_lyrics)

# Drop instrumentals and empty rows
before = len(df)
df = df.dropna(subset=['cleaned_lyrics'])
df = df[df['cleaned_lyrics'].str.strip() != '']
print(f"Removed {before - len(df)} instrumental/empty songs")
print(f"Remaining: {len(df)} songs")
df[['song_id', 'cleaned_lyrics']].head()

Removed 0 instrumental/empty songs
Remaining: 99242 songs


,song_id,cleaned_lyrics
0,QPozpoSeoWPWUp5k,together stars forever together stars forever ...
1,7AbZpDQ5co4eiTyc,last train nearly due underground closing soon...
2,SazEmQ5Ls4WqhvGE,takes main attraction scattered bones undernea...
3,mcSyQWsHtskCQIBJ,turn lights lock door fan flames know ready ke...
5,keJOOL6BeyRCG92K,quando voc chegar cho sentir que valeu quando ...


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
tfidf_matrix = vectorizer.fit_transform(df['cleaned_lyrics'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# Fast vectorized keyword extraction
feature_names = vectorizer.get_feature_names_out()

def get_top_keywords_fast(tfidf_row, top_n=10):
    indices = np.argsort(tfidf_row.data)[-top_n:]
    return [feature_names[tfidf_row.indices[i]] for i in indices[::-1]]

df['keywords'] = [get_top_keywords_fast(tfidf_matrix.getrow(i)) 
                  for i in range(len(df))]

print("Done!")
df[['song_id', 'keywords']].head()

TF-IDF matrix shape: (99242, 5000)
Done!


,song_id,keywords
0,QPozpoSeoWPWUp5k,"[stars, forever, together, next, saturday nigh..."
1,7AbZpDQ5co4eiTyc,"[restless, pocket, holds, train, shadows, with..."
2,SazEmQ5Ls4WqhvGE,"[nowhere, kids, ignorance, wished, innocence, ..."
3,mcSyQWsHtskCQIBJ,"[angel, sleeping, lies, melting, fan, turn lig..."
5,keJOOL6BeyRCG92K,"[quero, estar, seu, voc, pra, quando, aonde, f..."


In [12]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

print("Generating embeddings... this will take a while.")
print(f"Total songs: {len(df)}")

embeddings = model.encode(
    df['cleaned_lyrics'].tolist(),
    batch_size=64,
    show_progress_bar=True
)

print(f"Embeddings shape: {embeddings.shape}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generating embeddings... this will take a while.
Total songs: 99242


Batches:   0%|          | 0/1551 [00:00<?, ?it/s]

Embeddings shape: (99242, 384)


In [13]:
import pickle
import numpy as np

os.makedirs("../outputs", exist_ok=True)

# Save preprocessed lyrics
df[['song_id', 'lyrics', 'cleaned_lyrics', 'keywords']].to_csv(
    "../outputs/lyrics_preprocessed.csv", index=False
)

# Save TF-IDF matrix and vectorizer
with open("../outputs/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(tfidf_matrix, f)

with open("../outputs/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

# Save embeddings
np.save("../outputs/embeddings.npy", embeddings)

print("All outputs saved!")
print(f"  lyrics_preprocessed.csv — {len(df)} rows")
print(f"  tfidf_matrix.pkl        — {tfidf_matrix.shape}")
print(f"  tfidf_vectorizer.pkl    — {len(vectorizer.get_feature_names_out())} features")
print(f"  embeddings.npy          — {embeddings.shape}")

All outputs saved!
  lyrics_preprocessed.csv — 99242 rows
  tfidf_matrix.pkl        — (99242, 5000)
  tfidf_vectorizer.pkl    — 5000 features
  embeddings.npy          — (99242, 384)


In [1]:
import pandas as pd

# Load your preprocessed lyrics
df = pd.read_csv("../outputs/lyrics_preprocessed.csv")
print(f"Lyrics dataset: {len(df)} songs")

# Load the genre metadata from group 3
genres_df = pd.read_csv(
    "/Users/macbook_air/Music-Maven/group3/Sonic_fingerprinting_Apoorva_V01054205/Output_files/music4all/id_genres.csv",
    sep="\t"
)
genres_df.columns = ["song_id", "genre"]
print(f"Genre metadata: {len(genres_df)} songs")

# Merge
df_merged = df.merge(genres_df, on="song_id", how="left")
matched = df_merged["genre"].notna().sum()
print(f"Matched with genre labels: {matched} / {len(df_merged)}")
print(df_merged["genre"].value_counts().head(10))

# Save
df_merged.to_csv("../outputs/lyrics_preprocessed_with_genres.csv", index=False)
print("Saved!")


Lyrics dataset: 99242 songs
Genre metadata: 109269 songs
Matched with genre labels: 99242 / 99242
genre
pop                  6053
rock                 1839
rap                  1420
soul                 1320
indie rock           1150
electronic           1006
folk                  972
indie rock,rock       834
rap,hip hop           824
singer-songwriter     788
Name: count, dtype: int64
Saved!
